In [3]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_poisson_deviance
from utils import *
from scipy.stats import ttest_rel
from sklearn.metrics import d2_tweedie_score

X, y, exposure = load_data()
X = X.drop(columns=EXCLUDED_COLS)
X_train, X_test, y_train, y_test, exp_train, exp_test = train_test_split(
    X, y, exposure, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

feature_sets = {
    "traditional": [c for c in TRADITIONAL_COLS if c not in EXCLUDED_COLS],
    "telematics":  [c for c in X.columns if c not in TRADITIONAL_COLS],
    "combined":    list(X.columns),
}

# compare by CV, training data only
results = {}
for name, cols in feature_sets.items():
    alpha, folds = evaluate(X_train[cols], y_train, exp_train)
    results[name] = {"alpha": alpha, "folds": folds}
    print(f"{name}: alpha={alpha:.4g}, mean deviance={folds.mean():.4f}")

# paired check against combined
baseline = "combined"
base = results[baseline]["folds"]
rows = []
for name, r in results.items():
    if name == baseline:
        continue
    gap = r["folds"] - base
    t, p = ttest_rel(r["folds"], base)
    rows.append({
        "model": name,
        "mean_dev": r["folds"].mean(),
        "gap_vs_combined": gap.mean(),
        "gap_pct": 100 * gap.mean() / base.mean(),
        "gap_min": gap.min(),
        "gap_max": gap.max(),
        "folds_worse_than_baseline": f"{(gap > 0).sum()}/{len(gap)}",
        "t": t,
        "p": p,
    })
pd.DataFrame(rows).round(4)



traditional: alpha=0.001, mean deviance=0.3117
telematics: alpha=0.001, mean deviance=0.2855
combined: alpha=0.0003162, mean deviance=0.2789


,model,mean_dev,gap_vs_combined,gap_pct,gap_min,gap_max,folds_worse_than_baseline,t,p
0,traditional,0.3117,0.0328,11.7749,0.0295,0.0366,5/5,21.6024,0.0000
1,telematics,0.2855,0.0066,2.3763,0.0051,0.0076,5/5,12.6594,0.0002


In [4]:
# test set
cols = feature_sets["combined"]
cat_cols = X.select_dtypes(exclude="number").columns.tolist()
pipe = build_pipeline(cat_cols, results["combined"]["alpha"])
pipe.fit(X_train[cols], y_train / exp_train, sample_weight=exp_train)
print("Test deviance:", mean_poisson_deviance(y_test / exp_test, pipe.predict(X_test[cols]), sample_weight=exp_test))
pred = pipe.predict(X_test[cols])
d2 = d2_tweedie_score(y_test / exp_test, pred, power=1, sample_weight=exp_test)
print(f"Test D2: {d2:.6f}\n")
coefs = pd.Series(pipe[-1].coef_, index=pipe[:-1].get_feature_names_out())
print(coefs.sort_values(key=abs, ascending=False))

Test deviance: 0.2807329089391647
Test D2: 0.135333

Annual.pct.driven         0.596408
Car.age                  -0.265875
Credit.score             -0.236426
Pct.drive.thr             0.194310
Total.miles.driven        0.187318
Pct.drive.rush am        -0.166480
Brake.06miles             0.163801
Pct.drive.mon             0.152273
Pct.drive.tue             0.131262
Years.noclaims           -0.120172
Pct.drive.fri             0.098181
Accel.06miles             0.096208
Pct.drive.wed             0.080525
Pct.drive.sat             0.074003
Car.use_Farmer           -0.065772
Insured.age               0.065198
Avgdays.week              0.064745
Region_Urban              0.063351
Pct.drive.rush pm         0.053221
Annual.miles.drive        0.051269
Pct.drive.2hrs            0.048194
Left.turn.intensity08     0.037342
Car.use_Commute          -0.022603
Right.turn.intensity08    0.020098
Insured.sex_Male          0.018599
Car.use_Private          -0.017520
Marital_Single            0.017241
dt

# Experiment to test out if dropping territory improves the model

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_poisson_deviance
from utils import *
from scipy.stats import ttest_rel
from sklearn.metrics import d2_tweedie_score

X, y, exposure = load_data()
X_train, X_test, y_train, y_test, exp_train, exp_test = train_test_split(
    X, y, exposure, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

feature_sets = {
    "traditional_no_territory": [col for col in TRADITIONAL_COLS if col != "Territory"],
    "traditional": TRADITIONAL_COLS,
    # "telematics":  [c for c in X.columns if c not in TRADITIONAL_COLS],
    # "combined":    list(X.columns),
}

# compare by CV, training data only
results = {}
for name, cols in feature_sets.items():
    alpha, folds = evaluate(X_train[cols], y_train, exp_train)
    results[name] = {"alpha": alpha, "folds": folds}
    print(f"{name}: alpha={alpha:.4g}, mean deviance={folds.mean():.4f}")

# paired check against combined
baseline = "traditional"
base = results[baseline]["folds"]
rows = []
for name, r in results.items():
    if name == baseline:
        continue
    gap = r["folds"] - base
    t, p = ttest_rel(r["folds"], base)
    rows.append({
        "model": name,
        "mean_dev": r["folds"].mean(),
        "gap_vs_baseline": gap.mean(),
        "gap_pct": 100 * gap.mean() / base.mean(),
        "gap_min": gap.min(),
        "gap_max": gap.max(),
        "folds_worse_than_baseline": f"{(gap > 0).sum()}/{len(gap)}",
        "t": t,
        "p": p,
    })
pd.DataFrame(rows).round(4)



traditional_no_territory: alpha=0.001, mean deviance=0.3117
traditional: alpha=0.003162, mean deviance=0.3125


,model,mean_dev,gap_vs_baseline,gap_pct,gap_min,gap_max,folds_worse_than_baseline,t,p
0,traditional_no_territory,0.3117,-0.0007,-0.2348,-0.0015,-0.0002,0/5,-3.2953,0.0301


Based on the result above, we are going to drop the 'Territory' column as the mean poisson deviation is lower after dropping the column across all folds.